# Rotator- Torque Analysis and Rotator position

<b> Associated tickets: </b> <br>
- [SITCOM-1884] https://rubinobs.atlassian.net/browse/SITCOM-1884)
- [SITCOM-1818]https://rubinobs.atlassian.net/browse/SITCOM-1818) : Similar, but the analysis from ComCam on Sky

<b> Description </b>

We performed full-range movements with the rotator twice this week. 

- On Feb 25, 2025, we ran movements using only the Rotator. 
- On Feb 27, 2025, we ran movements using the Rotator and the Camera Hexapod. 

In principle, the Rotator should not see any extra torque even with the VIP lines installed. All the load of these lines should go to the camera cable wrap. Is that the case? We might need to compare data with ComCam on Sky campaign.

In [ ]:
#%matplotlib widget
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from astropy.time import Time
from pathlib import Path
from datetime import datetime

from lsst.summit.utils.tmaUtils import (
    TMAEventMaker,
    TMAState,
)
from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData

In [ ]:
plot_path = Path("./plots_LSSTCam")
plot_path.mkdir(exist_ok=True, parents=True)

event_maker = TMAEventMaker()
efd_client = makeEfdClient()

In [ ]:
unit = r'N$\cdot$m'

# Data Analysis

Rotator Soak Tests were proceeded in the following time windows:
- From <code>2025-02-25 17:34 - 16:00</code>: first full range rotation tests without hexapod movements.
- From <code>2025-02-27 13:00 - 19:00</code>: LVV-T2261 - rotator and camera hexapod movements. 
- From <code>2025-02-27 23:00 - 2025-02-28 02:00</code>: BLOCK-T358 - small soak test using rotator and camera hexapod.
- From <code>2025-02-28 00:01 - 01:38 UTC</code>
- From <code>2025-02-28 17:38 - 17:49 UTC</code>
- From <code>2025-02-28 17:57 - 18:02 UTC</code>
- From <code>2025-03-01 13:35 - 15:54 UTC</code>

## Test #1 


In [ ]:
start_time_d1 = "2025-02-25T17:34:00"
end_time_d1 = "2025-02-25T18:45:00"

In [ ]:
start_time= Time(start_time_d1, scale="utc", format="isot")
end_time = Time(end_time_d1, scale="utc", format="isot")

In [ ]:
rot = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTRotator.rotation",
    columns=["actualPosition"],
    begin=start_time,
    end= end_time
)

In [ ]:
rot_motor = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTRotator.motors",
    columns=["torque0","torque1"],
    begin=start_time,
    end= end_time
)

In [ ]:
dayobs="2025-02-25"
fig, ax = plt.subplots(num=f"day_analysis_{dayobs}",figsize=(10, 5))

max_torque0 = np.max(np.abs(rot_motor["torque0"]))
mean_torque0= np.mean((abs(rot_motor["torque0"])))      
max_torque1 = np.max(np.abs(rot_motor["torque1"]))
mean_torque1= np.mean((abs(rot_motor["torque1"])))

text= [
    f"Max:{max_torque0:.1e}, Mean:{mean_torque0:.1e} (unit: {unit})",
    f"Max:{max_torque1:.1e}, Mean:{mean_torque1:.1e} (unit: {unit})",
]

line1, = ax.plot(rot_motor.index, rot_motor["torque0"],color="C0",label=f"torque0 {text[0]}")
line2, = ax.plot(rot_motor.index, rot_motor["torque1"],color="C1",label=f"torque1 {text[1]}")
ax.set_xlabel("Time [UTC]")
ax.set_ylabel(r"Torque (N$\cdot$m)")
ax2 = ax.twinx()
line3, = ax2.plot(rot.index, rot["actualPosition"],color="C2",label="Rotator Angle")
ax2.set_ylabel("Rotator Angle (deg)", rotation=270, labelpad=15)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
fig.suptitle(f"Rotator Torque Analysis with Pancake Wrap {dayobs}: Rotator only")
ax.grid(which="major", axis="both", linestyle="--")
handles = [line1, line2, line3]
    
ax2.legend(handles=handles, loc="lower left", fontsize=11)
    # fig.tight_layout()
#ax.set_xticks(ax.get_xticks().tolist())
#ax.set_xticklabels([ts for ts in x_major_positions])
fig.savefig(plot_path / f"{dayobs}_Rot_only.png")
plt.show()

## Test #2

In [ ]:
start_time_d2 = "2025-02-27T13:50:00"
end_time_d2 = "2025-02-27T18:00:16"

In [ ]:
start_time= Time(start_time_d2, scale="utc", format="isot")
end_time = Time(end_time_d2, scale="utc", format="isot")

In [ ]:
rot_motor = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTRotator.motors",
    columns=["torque0", "torque1"],
    begin=start_time,
    end= end_time
)

In [ ]:
rot = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTRotator.rotation",
    columns=["actualPosition"],
    begin=start_time,
    end= end_time
)

In [ ]:
dayobs="2025-02-27"
fig, ax = plt.subplots(num=f"day_analysis_{dayobs}",figsize=(10, 5))


max_torque0 = np.max(np.abs(rot_motor["torque0"]))
mean_torque0= np.mean((abs(rot_motor["torque0"])))      
max_torque1 = np.max(np.abs(rot_motor["torque1"]))
mean_torque1= np.mean((abs(rot_motor["torque1"])))

text= [
    f"Max:{max_torque0:.1e}, Mean:{mean_torque0:.1e} (unit: {unit})",
    f"Max:{max_torque1:.1e}, Mean:{mean_torque1:.1e} (unit: {unit})",
]

line1, = ax.plot(rot_motor.index, rot_motor["torque0"],color="C0",label=f"torque0 {text[0]}")
line2, = ax.plot(rot_motor.index, rot_motor["torque1"],color="C1",label=f"torque1 {text[1]}")
ax.set_xlabel("Time [UTC]")
ax.set_ylabel(r"Torque (N$\cdot$m)")

ax2 = ax.twinx()
line3, = ax2.plot(rot.index, rot["actualPosition"],color="C2",label="Rotator Angle")
ax2.set_ylabel("Rotator Angle (deg)", rotation=270, labelpad=15)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
fig.suptitle(f"LVV-T2261 - rotator and camera hexapod movements {dayobs}")
ax.grid(which="major", axis="both", linestyle="--")
handles = [line1, line2, line3]
    
ax2.legend(handles=handles, loc="lower left", fontsize=11)

    # fig.tight_layout()
#ax.set_xticks(ax.get_xticks().tolist())
#ax.set_xticklabels([ts for ts in x_major_positions])
plt.show()
fig.savefig(plot_path / f"{dayobs}_LVV-T2261.png")

## Test #3

In [ ]:
start_time_d3 = "2025-02-27T23:30:00"
end_time_d3 = "2025-02-28T01:45:00"

In [ ]:
start_time= Time(start_time_d3, scale="utc", format="isot")
end_time = Time(end_time_d3, scale="utc", format="isot")

In [ ]:
rot_motor = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTRotator.motors",
    columns=["torque0", "torque1"],
    begin=start_time,
    end= end_time
)

In [ ]:
rot = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTRotator.rotation",
    columns=["actualPosition"],
    begin=start_time,
    end= end_time
)

In [ ]:
dayobs="2025-02-27"
fig, ax = plt.subplots(num=f"day_analysis_{dayobs}-2025-02-28",figsize=(10, 5))


max_torque0 = np.max(np.abs(rot_motor["torque0"]))
mean_torque0= np.mean((abs(rot_motor["torque0"])))      
max_torque1 = np.max(np.abs(rot_motor["torque1"]))
mean_torque1= np.mean((abs(rot_motor["torque1"])))

text= [
    f"Max:{max_torque0*(10**6):.1f}, Mean:{mean_torque0*(10**6):.1f} (unit: {unit})",
    f"Max:{max_torque1*(10**6):.1f}, Mean:{mean_torque1*(10**6):.1f} (unit: {unit})",
]


line1, = ax.plot(rot_motor.index, rot_motor["torque0"],color="C0",label=f"torque0 {text[0]}")
line2, = ax.plot(rot_motor.index, rot_motor["torque1"],color="C1",label=f"torque1 {text[1]}")
ax.set_xlabel("Time [UTC]")
ax.set_ylabel(r"Torque (N$\cdot$m)")

ax2 = ax.twinx()
line3, = ax2.plot(rot.index, rot["actualPosition"],color="C2",label="Rotator Angle")
ax2.set_ylabel("Rotator Angle (deg)", rotation=270, labelpad=15)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
fig.suptitle(f"BLOCK-T358 - small soak test using rotator and camera hexapod {dayobs}")
ax.grid(which="major", axis="both", linestyle="--")
handles = [line1, line2, line3]
    
ax2.legend(handles=handles, loc="lower left", fontsize=11)

    # fig.tight_layout()
#ax.set_xticks(ax.get_xticks().tolist())
#ax.set_xticklabels([ts for ts in x_major_positions])
plt.show()
fig.savefig(plot_path / f"{dayobs}_BLOCK-T358.png")

## Test #4

In [ ]:
start_time_d4 = "2025-02-28T17:38:43"
end_time_d4 = "2025-02-28T17:49:03"

In [ ]:
start_time= Time(start_time_d4, scale="utc", format="isot")
end_time = Time(end_time_d4, scale="utc", format="isot")

In [ ]:
rot_motor = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTRotator.motors",
    columns=["torque0", "torque1"],
    begin=start_time,
    end= end_time
)

In [ ]:
rot = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTRotator.rotation",
    columns=["actualPosition"],
    begin=start_time,
    end= end_time
)

In [ ]:
dayobs="2025-02-28"
fig, ax = plt.subplots(num=f"day_analysis_{dayobs}",figsize=(10, 5))


max_torque0 = np.max(np.abs(rot_motor["torque0"]))
mean_torque0= np.mean((abs(rot_motor["torque0"])))      
max_torque1 = np.max(np.abs(rot_motor["torque1"]))
mean_torque1= np.mean((abs(rot_motor["torque1"])))

text= [
    f"Max:{max_torque0:.1e}, Mean:{mean_torque0:.1e} (unit: {unit})",
    f"Max:{max_torque1:.1e}, Mean:{mean_torque1:.1e} (unit: {unit})",
]

line1, = ax.plot(rot_motor.index, rot_motor["torque0"],color="C0",label=f"torque0 {text[0]}")
line2, = ax.plot(rot_motor.index, rot_motor["torque1"],color="C1",label=f"torque1 {text[1]}")
ax.set_xlabel("Time [UTC]")
ax.set_ylabel(r"Torque (N$\cdot$m)")

ax2 = ax.twinx()
line3, = ax2.plot(rot.index, rot["actualPosition"],color="C2",label="Rotator Angle")
ax2.set_ylabel("Rotator Angle (deg)", rotation=270, labelpad=15)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
fig.suptitle(f"Rotator Soak Test#4 {dayobs}")
ax.grid(which="major", axis="both", linestyle="--")
handles = [line1, line2, line3]
    
ax2.legend(handles=handles, loc="lower left", fontsize=12)

    # fig.tight_layout()
#ax.set_xticks(ax.get_xticks().tolist())
#ax.set_xticklabels([ts for ts in x_major_positions])
plt.show()
fig.savefig(plot_path / f"{dayobs}_Test4.png")

## Test #5

In [ ]:
start_time_d5 = "2025-02-28T17:57:50"
end_time_d5 = "2025-02-28T18:02:21"

In [ ]:
start_time= Time(start_time_d5, scale="utc", format="isot")
end_time = Time(end_time_d5, scale="utc", format="isot")

In [ ]:
rot_motor = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTRotator.motors",
    columns=["torque0", "torque1"],
    begin=start_time,
    end= end_time
)

In [ ]:
rot = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTRotator.rotation",
    columns=["actualPosition"],
    begin=start_time,
    end= end_time
)

In [ ]:
dayobs="2025-02-28"
fig, ax = plt.subplots(num=f"day_analysis_{dayobs}",figsize=(10, 5))


max_torque0 = np.max(np.abs(rot_motor["torque0"]))
mean_torque0= np.mean((abs(rot_motor["torque0"])))      
max_torque1 = np.max(np.abs(rot_motor["torque1"]))
mean_torque1= np.mean((abs(rot_motor["torque1"])))

text= [
    f"Max:{max_torque0:.1e}, Mean:{mean_torque0:.1e} (unit: {unit})",
    f"Max:{max_torque1:.1e}, Mean:{mean_torque1:.1e} (unit: {unit})",
]


line1, = ax.plot(rot_motor.index, rot_motor["torque0"],color="C0",label=f"torque0 {text[0]}")
line2, = ax.plot(rot_motor.index, rot_motor["torque1"],color="C1",label=f"torque1 {text[1]}")
ax.set_xlabel("Time [UTC]")
ax.set_ylabel(r"Torque (N$\cdot$m)")

ax2 = ax.twinx()
line3, = ax2.plot(rot.index, rot["actualPosition"],color="C2",label="Rotator Angle")
ax2.set_ylabel("Rotator Angle (deg)", rotation=270, labelpad=15)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
fig.suptitle(f"Rotator Soak Test#5 {dayobs}")
ax.grid(which="major", axis="both", linestyle="--")
handles = [line1, line2, line3]
    
ax2.legend(handles=handles, loc="lower left", fontsize=11)

# fig.tight_layout()
#ax.set_xticks(ax.get_xticks().tolist())
#ax.set_xticklabels([ts for ts in x_major_positions])
plt.show()
fig.savefig(plot_path / f"{dayobs}_Test5.png")

## Test #6

In [ ]:
start_time_d6 = "2025-03-01T13:35:05"
end_time_d6 = "2025-03-01T15:54:36"

In [ ]:
start_time= Time(start_time_d6, scale="utc", format="isot")
end_time = Time(end_time_d6, scale="utc", format="isot")

In [ ]:
rot_motor = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTRotator.motors",
    columns=["torque0", "torque1"],
    begin=start_time,
    end= end_time
)

In [ ]:
rot = getEfdData(
    client=efd_client,
    topic="lsst.sal.MTRotator.rotation",
    columns=["actualPosition"],
    begin=start_time,
    end= end_time
)

In [ ]:
dayobs="2025-03-01"
fig, ax = plt.subplots(num=f"day_analysis_{dayobs}",figsize=(10, 5))


max_torque0 = np.max(np.abs(rot_motor["torque0"]))
mean_torque0= np.mean((abs(rot_motor["torque0"])))      
max_torque1 = np.max(np.abs(rot_motor["torque1"]))
mean_torque1= np.mean((abs(rot_motor["torque1"])))

text= [
    f"Max:{max_torque0:.1e}, Mean:{mean_torque0:.1e} (unit: {unit})",
    f"Max:{max_torque1:.1e}, Mean:{mean_torque1:.1e} (unit: {unit})",
]


line1, = ax.plot(rot_motor.index, rot_motor["torque0"],color="C0",label=f"torque0 {text[0]}")
line2, = ax.plot(rot_motor.index, rot_motor["torque1"],color="C1",label=f"torque1 {text[1]}")
ax.set_xlabel("Time [UTC]")
ax.set_ylabel(r"Torque (N$\cdot$m)")

ax2 = ax.twinx()
line3, = ax2.plot(rot.index, rot["actualPosition"],color="C2",label="Rotator Angle")
ax2.set_ylabel("Rotator Angle (deg)", rotation=270, labelpad=15)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
fig.suptitle(f"Rotator Soak Test#6 {dayobs}")
ax.grid(which="major", axis="both", linestyle="--")
handles = [line1, line2, line3]
    
ax2.legend(handles=handles, loc="lower left", fontsize=11)

    # fig.tight_layout()
#ax.set_xticks(ax.get_xticks().tolist())
#ax.set_xticklabels([ts for ts in x_major_positions])
plt.show()
fig.savefig(plot_path / f"{dayobs}_Test6.png")